# Cellular Automata Playground

Before building a Neural Cellular Automaton, explore its basic mechanism:

$$\text{state} \rightarrow \text{perceive locally} \rightarrow \text{update} \rightarrow \text{repeat}$$

Draw on the grid and change both **perception** and **update**. Cell states may be binary or continuous values between 0 and 1.

In [ ]:
from html import escape
from IPython.display import HTML, display

ca_playground = r"""
<!doctype html><html><head><meta charset="utf-8"><style>
*{box-sizing:border-box}body{margin:0;font-family:system-ui,sans-serif;color:#202124}.app{max-width:1100px;margin:auto;padding:14px;border:1px solid #dadce0;border-radius:14px;background:#f8f9fa}.top,.row,.stats{display:flex;gap:8px;align-items:center;flex-wrap:wrap}.top{margin-bottom:12px}.layout{display:grid;grid-template-columns:minmax(360px,1fr) 340px;gap:14px}.canvas-wrap{background:#222;border-radius:10px;padding:8px;display:flex}.canvas-wrap canvas{width:100%;aspect-ratio:1;cursor:crosshair;touch-action:none}.panel{display:flex;flex-direction:column;gap:10px;max-height:680px;overflow:auto}.card{background:#fff;border:1px solid #dadce0;border-radius:10px;padding:11px}.card h3{font-size:15px;margin:0 0 8px}button,select,input{font:inherit;border:1px solid #bdc1c6;border-radius:7px;background:#fff;padding:6px 8px}button{cursor:pointer;font-weight:600}.primary{background:#333;color:#fff!important}.grow{flex:1}.wide{width:100%}label{font-size:13px;font-weight:600}.kernel{display:grid;grid-template-columns:repeat(3,52px);gap:4px;margin:8px 0}.kernel input{width:52px;text-align:center;padding:5px 2px}.small{font-size:12px;color:#5f6368;line-height:1.4}.formula{font-family:monospace;font-size:12px}.stats{justify-content:space-between;margin-top:8px;font-size:13px}.hidden{display:none}.error{color:#b3261e;font-weight:600}.swatch{height:9px;border-radius:5px;background:linear-gradient(90deg,#000,#fff);border:1px solid #aaa}@media(max-width:800px){.layout{grid-template-columns:1fr}.panel{max-height:none}}
</style></head><body><div class="app"><div class="top">
<button id="start" class="primary">▶ Start</button><button id="step">Step</button><button id="reset">Reset</button><button id="clear">Clear</button><label>Speed <input id="speed" type="range" min="1" max="30" value="8"></label></div>
<div class="layout"><div><div class="canvas-wrap"><canvas id="grid" width="640" height="640"></canvas></div><div class="stats"><span id="stats"></span><span id="message">Click or drag to draw.</span></div></div><div class="panel">
<div class="card"><h3>Initial state</h3><div class="row"><select id="initial" class="grow"><option>Single impulse</option><option>Glider</option><option>Random binary</option><option>Random continuous</option><option>Image</option></select><button id="load">Load</button></div><label>Brush value: <span id="brushLabel">1.00</span></label><input id="brush" class="wide" type="range" min="0" max="1" step="0.05" value="1"><div class="swatch"></div><div class="small">Left-click paints this value; right-click erases.</div></div>
<div class="card"><h3>1 · Perceive</h3><div class="row"><label>Preset <select id="perceptionPreset"><option value="neighbors">Neighbors</option><option value="average">Average</option><option value="laplace">Laplacian</option><option value="sobel">Sobel-x</option></select></label></div><div class="small">Weighted sum of the 3×3 neighborhood:</div><div id="kernel" class="kernel"></div><label><input id="normalize" type="checkbox"> Normalize by |weights|</label></div>
<div class="card"><h3>2 · Update</h3><div class="row"><label>Preset <select id="updatePreset"><option value="life">Game of Life</option><option value="diffusion">Diffusion</option><option value="growth">Threshold growth</option><option value="oscillating">Oscillating response</option><option value="tent">Tent map</option><option value="custom">Custom</option></select></label></div><label>Rule type <select id="ruleType"><option value="life">Life-like (binary)</option><option value="formula">Continuous formula</option></select></label><div id="life"><div class="row"><label>Birth <input id="born" value="3" size="7"></label><label>Survive <input id="survive" value="2,3" size="7"></label></div><div class="small">Comma-separated perceived values.</div></div><div id="continuous" class="hidden"><label>New state =</label><input id="formula" class="wide formula" value="x + 0.25 * (p - x)"><div class="small"><b>x</b>: current value · <b>p</b>: perception<br>Functions: clip, abs, sin, cos, exp, min, max</div><label><input id="clamp" type="checkbox" checked> Clamp to [0,1]</label></div></div>
</div></div></div><script>
const N=48,C=document.getElementById('grid'),X=C.getContext('2d');let g=blank(),gen=0,running=false,timer,drawing=false,paintValue=1;const PORTRAIT='Gh4iJCIiHx4eHh0bHBkZHBwaFxcZGBocHB4gIRwcHx8bGxweGxocHB4fHhwdGxgXHSIkISEjIiEhISAhHxsbHB0aFxcXGBkYGhsfHiEhIB8fHx0eHRwcGxobIB8hHxgWHiEiICEnJiIjIyEfHxsZGhwYFxgZGhobGBogHRYXGx0hIyEgHh4bHB8jIh4bHBcWICEjJCMoKCQiICEhHBsaGhoZGBkcHh8eHSEaLVVdRTUlICUkHyAhISIjHx8aHBUXICMnJSUlJSUlIyIfHR8eGxgaGxweHx8eGxRFk5mnyb6fVyghICAhICEgHh8bHhgYHyElJSMmJSUmJiUkJCMhICAhHyAjIx4bLlyvupun1N/gqGU1ISEgISIhIyIgIBgYHiImJicoJycoKCYnJicmIyMiIh4aKDxfp8XO3s20r6eqt2hIMyMhICIkIyAgHxYYHiIlJyonKSooKCcnJSUmJCMiHStklq+tpp6MhLLJr6GOoZZZPiwhICAkIiAeHhgaHiEjJSgoKCorKSgqJiQkIiYgNKDIw7qsqaqjim1/p7KTgoh4SDQiHyMjISAdHRocHSEjJygoKSorKispJSQiIyU0rMa6vr6vo6OgnZZ6YYCgdF55VD0kHiAiIB4bGBcYICMjJSgoKisrKSkoJyMgJC2Qxb+/vbWtpJ+glpiZaEhfg1J9YDopHyIgHx8bGxYWIiQlJiYpKiwrKCYmKCUlHm3Fub24tbCmpqWcmpeQb1dDUmCLfEYwHB4eHhwdHhgXICMnKScpKiopKCYmKCckMZ6ssLe2tpt/mqCjiI9/SldVS0mdilM/FhsbGxsfHRoYISInKy0rKCkqKSkoJycjOqHN08WrpY58c3WQaU9fQEJPTUOMiFFYHhIYGx0dGxkZHyEkKiopJycpJiknKigdSdXj4dvKroqUck9iVzdFTEVKUkB9hkxlLhgZHBsdHBsXHh8jJycmJyYlKCopKCkekObb2tLRzaZ3X0FHS0dEVU5GST1tjEthMiEXGhocHRwUHh8gISQlJiUkJikpKiYsoMjd1cfBwsGlaUc5Lzs7P0dNT0JekktnNB8YGBwbHBkXHiEgISIjJCMkJigoLCsmcpO8zKGcq7u4jlJKOjg/LzZGT0JOm095QyMfHh4dHBwZHyEhJSQkIyYlJSUnLS0gkXqhwoqvsnOZfVZJS0k9OTNCSjo9pllzWjEgIB8eHh0aHSEkJyQlJSUkJCUnLSwusbvTqpDUsztbamFUR09APD05PzMxo2VnZTUgHSAeHRwcGyEkJiUmJigkHSEnLCk5y9jWmpzGsISHon5ZUUpIPENAOy4ro4teaT4mFxwbHRwaGSEjICQmKCgkIiEkKSY4xtu8h5jDwbu5qX1VVlFISUQ9OTIomphfZEkpFhscHRwaGSIlHyEkJyckIyMgJikvvNefd43ExMKylGZSVVNKPEw9LSopjp9hbUcpFRsdGx0ZHCIlHxojIiUmIyIdIisiqctmZmm3wbahgVhWWFdMSkUwJCgjhaRlhkc/HRoeGxwZHSAeHBwiIyYlISEeICcchuedZXyxu6yXdVhZWVRWUzIgJygcgq1prU5VJBwcGxsaGx4bGxogJCooJCMgHyUiXcaskaWur5+Kb2JeW09IOSciJCYclslkxFpdLhwdGh0cHCQhIBwgJSonJSQiIicqNqB4fXONo5F8b3BsY1NMPSwlIyMcj9Box3BaRxweHR0aICQkIx4iJSQjJiUjIycsI369hWyVpIl7dHh8Z2CYRjEnICUahNh0vItVVSAdHiAbICUlJB0hJCUlJSYkJCYtJVDYpIWQmIeBgIZ3Xl97WC8zOEMwdMaHsKVPaSkeHx8cIiklJCQmJiYkJCQjIyMpKimh1qeMj4Z/gnRcVlNdeJCgn4xgbbSio7VHdDYhIiAcIykpJiUlJiUjIiEjJCMmKykpYnloUmuBbG2EoL/i6s+UcmhNbberlb9HbUIjIyAbISIjICMiIyUlJCIiIiEgISQkKScnF1j06sGchmpbTkdIQT89NqK+l5x1XX4wJB8dHiAhHRseHiEiIiIhIiAeHiAkJCQfL+PlpaGklmhWRT5CPkJDLqDIq4VzT4VbJiEcHB0dGxsaHyAfHyEhICMhHx8iICYVfeqUp6qdfGBVQDg6QEFDNJfOsoiAWpyNKyAaGhsdHhwaICEcHiIhHyAhHh4dHSMXgKCXp6ief2ROQT4+QkVEMIvTsIyLbtGmNR8ZFxoeGRscHyAfISIiIB8fHB0bHhwrhJmkp6GYj2RHQkJBQUM/KX/LsIqScMa0TR8aFxodGBkbICMfHyAiHhwbHB0dIBhToKKZk5SbmoRMRkE8QEE2Jmy1xIyid7PIWh4bFxsdGBkbHyEiIB8gHB4cHR0dGiZ4lYmHlqiqqJlZUlRHQEE3JXG1xJKGcaGeNx8eGBkaGRcaHSEgHh0cHSAdHhsdGEyQioiVnKSgopZdUl5YUEU6JWmdpJU4JjImICMiGxwbHBoaHR4fHRsbHh0eIB0hG2aSj5qfopuWl5d0VFdUU09DK2uapIg5Ih8jJSUfHB0cHRsbGx0fHBocHR0eHx0hG2iZmqWnoJGPkpOKYU1OTUtOUj4cFh4fISEfIh8ZGxwdHR0bHR4eHBkZHRsfHx8ZS56Xr7SrjoCCg3+HdVFJSERMTU88IiEgICMdIB0XGh0fHh4aGxsbGBMXHB0eHyEgiZudqaOchHl7f3h/ckhDS0tOTkpKKR0hIiMeHRoXGR4fHhwcHBwdGhUVGBweHhpNoJegdX2Mf3ZzdXODcUZBTk1LU0RSOx0gISQdGxoWGRwfHRweHh4gIRsYFx0eHySDmaSaaneKgnVxbXiJeU5CUExJWkVMSyYcHyEcHBwX';const K={neighbors:[1,1,1,1,0,1,1,1,1],average:[1,1,1,1,1,1,1,1,1],laplace:[0,1,0,1,-4,1,0,1,0],sobel:[-1,0,1,-2,0,2,-1,0,1]};function blank(){return Array.from({length:N},()=>Array(N).fill(0))}function copy(a){return a.map(r=>r.slice())}const KB=document.getElementById('kernel');for(let i=0;i<9;i++){let e=document.createElement('input');e.type='number';e.step='.1';KB.appendChild(e)}function setKernel(n){[...KB.children].forEach((e,i)=>e.value=K[n][i]);normalize.checked=n==='average'}function kernel(){return[...KB.children].map(e=>{let v=Number(e.value);if(!Number.isFinite(v))throw Error('Kernel entries must be numbers.');return v})}function idx(i,m){if(m==='wrap')return(i+N)%N;if(m==='reflect')return Math.min(N-1,Math.max(0,i));return i<0||i>=N?null:i}
function perceive(a,r,c,k){let p=0,w=0;for(let dr=-1;dr<=1;dr++)for(let dc=-1;dc<=1;dc++){let q=k[(dr+1)*3+dc+1],rr=idx(r+dr,'wrap'),cc=idx(c+dc,'wrap');p+=q*a[rr][cc];w+=Math.abs(q)}return normalize.checked&&w?p/w:p}function setOf(id){let t=document.getElementById(id).value.trim();if(!t)return[];let a=t.split(/[ ,]+/).filter(Boolean).map(Number);if(a.some(v=>!Number.isFinite(v)))throw Error('Rule values must be numbers.');return a}const clip=v=>Math.min(1,Math.max(0,v));function update(a){let k=kernel(),out=blank(),life=ruleType.value==='life',B,S,F;if(life){B=setOf('born');S=setOf('survive')}else F=new Function('x','p','clip','abs','sin','cos','exp','min','max',`return (${formula.value})`);for(let r=0;r<N;r++)for(let c=0;c<N;c++){let x=a[r][c],p=perceive(a,r,c,k),v;if(life)v=x>.5?+S.some(q=>Math.abs(q-p)<1e-6):+B.some(q=>Math.abs(q-p)<1e-6);else{v=Number(F(x,p,clip,Math.abs,Math.sin,Math.cos,Math.exp,Math.min,Math.max));if(!Number.isFinite(v))throw Error('Formula returned a non-finite value.');if(clamp.checked)v=clip(v)}out[r][c]=v}return out}
function draw(){let d=X.createImageData(N,N);for(let r=0;r<N;r++)for(let c=0;c<N;c++){let v=Math.round(clip(g[r][c])*255),i=(r*N+c)*4;d.data[i]=d.data[i+1]=d.data[i+2]=v;d.data[i+3]=255}let off=document.createElement('canvas');off.width=off.height=N;off.getContext('2d').putImageData(d,0,0);X.imageSmoothingEnabled=false;X.clearRect(0,0,C.width,C.height);X.drawImage(off,0,0,C.width,C.height);let s=C.width/N;X.strokeStyle='rgba(120,120,120,.15)';X.beginPath();for(let i=0;i<=N;i++){X.moveTo(i*s,0);X.lineTo(i*s,C.height);X.moveTo(0,i*s);X.lineTo(C.width,i*s)}X.stroke();stats.textContent=`Generation ${gen}`}function msg(t,bad=false){message.textContent=t||'Click or drag to draw.';message.className=bad?'error':''}function advance(){try{g=update(g);gen++;draw();msg('')}catch(e){pause();msg(e.message,true)}}function schedule(){clearInterval(timer);if(running)timer=setInterval(advance,1000/Number(speed.value))}function pause(){running=false;clearInterval(timer);start.textContent='▶ Start'}function toggle(){running=!running;start.textContent=running?'Ⅱ Pause':'▶ Start';schedule()}
function load(){pause();g=blank();let select=document.getElementById('initial'),p=select.value,m=N>>1;if(p==='Glider')[[0,1],[1,2],[2,0],[2,1],[2,2]].forEach(([r,c])=>g[m-1+r][m-1+c]=1);if(p==='Image'){let bytes=atob(PORTRAIT),rows=bytes.length/N;for(let r=0;r<N;r++)for(let c=0;c<N;c++){let sourceRow=Math.min(rows-1,Math.floor(r*rows/N));g[r][c]=bytes.charCodeAt(sourceRow*N+c)/255}}if(p==='Random binary')for(let r=0;r<N;r++)for(let c=0;c<N;c++)g[r][c]=Math.random()<.25?1:0;if(p==='Random continuous')for(let r=0;r<N;r++)for(let c=0;c<N;c++)g[r][c]=Math.random();if(p==='Single impulse')g[m][m]=1;window.savedInitial=copy(g);gen=0;draw()}function controls(){let b=ruleType.value==='life';life.classList.toggle('hidden',!b);continuous.classList.toggle('hidden',b)}function markCustom(){document.getElementById('updatePreset').value='custom'}function setUpdatePreset(name){if(name==='life'){ruleType.value='life';born.value='3';survive.value='2,3'}else{ruleType.value='formula';if(name==='diffusion')formula.value='x + 0.25 * (p - x)';if(name==='growth')formula.value='clip(x + 0.12 * (p - 0.45))';if(name==='oscillating')formula.value='clip(x + 0.08 * sin(8 * p))';if(name==='tent')formula.value='clip(1 - abs(2 * p - 1))'}controls();msg('')}
function pos(e){let b=C.getBoundingClientRect();return[Math.floor((e.clientY-b.top)*N/b.height),Math.floor((e.clientX-b.left)*N/b.width)]}function paint(e){let[r,c]=pos(e);if(r>=0&&r<N&&c>=0&&c<N){g[r][c]=paintValue;window.savedInitial=copy(g);gen=0;draw()}}C.oncontextmenu=e=>e.preventDefault();C.onpointerdown=e=>{e.preventDefault();pause();drawing=true;paintValue=e.button===2?0:Number(brush.value);C.setPointerCapture(e.pointerId);paint(e)};C.onpointermove=e=>{if(drawing)paint(e)};C.onpointerup=()=>drawing=false;document.getElementById('start').onclick=toggle;document.getElementById('step').onclick=()=>{pause();advance()};document.getElementById('reset').onclick=()=>{pause();g=copy(window.savedInitial);gen=0;draw()};document.getElementById('clear').onclick=()=>{pause();g=blank();window.savedInitial=copy(g);gen=0;draw()};document.getElementById('load').onclick=load;speed.oninput=schedule;brush.oninput=()=>brushLabel.textContent=Number(brush.value).toFixed(2);perceptionPreset.onchange=()=>{setKernel(perceptionPreset.value);draw()};updatePreset.onchange=()=>setUpdatePreset(updatePreset.value);ruleType.onchange=()=>{markCustom();controls()};[...KB.children,normalize].forEach(e=>e.oninput=()=>msg(''));[born,survive,formula,clamp].forEach(e=>e.oninput=()=>{markCustom();msg('')});setKernel('neighbors');normalize.checked=false;setUpdatePreset('life');load();
</script></body></html>
"""
display(HTML(f'<iframe srcdoc="{escape(ca_playground, quote=True)}" width="100%" height="790" style="border:0"></iframe>'))

### Things to try

- Try different initial states by drawing directly on the grid or loading one of the presets.
- Modify one Birth or Survival value in Conway's rule.
- Replace neighbor counting with an Average or Laplacian perception kernel.
- Switch to a continuous update and draw with a gray brush value.
- Pause and use **Step**: can you explain a cell's next value from its neighborhood?

The pattern is always **perceive locally → update locally → repeat globally**. Next, we replace the handcrafted update with a learnable neural network.

# From CA to Neural CA

In the playground, we designed the local rule and observed the resulting global behavior. We now reverse the problem:

> We specify a desired global outcome and train a neural network to discover a local update rule.

This is deliberately a small experiment: the goal is to understand the mechanism, not to maximize performance.

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

GRID_SIZE = 48  # Fixed to match the CA playground.
torch.manual_seed(7)
np.random.seed(7)
random.seed(7)
torch.set_num_threads(min(4, torch.get_num_threads()))  # Keeps the tiny CPU workload responsive.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using {device}")

## Experiment settings

This is the central place for changing the model and training setup. Start with the defaults, then change one value and rerun the NCA section.

In [ ]:
STATE_CHANNELS = 8          # One visible channel plus seven hidden channels.
HIDDEN_SIZE = 32            # Width of the neural update network.
TARGET_PRESET = "image"   # Choose: "image", "circle", or "cross".

TRAINING_ITERATIONS = 200   # Increase for quality; decrease for a faster experiment.
TRAINING_ROLLOUT_STEPS = 32 # Number of NCA updates before computing the loss.
RANDOMIZE_ROLLOUT_STEPS = False  # Set True to sample the rollout length below.
ROLLOUT_STEP_RANGE = (24, 40)    # Used only when randomization is enabled.


## 1. Choose the desired global outcome

The default `image` target uses the same grayscale portrait as the CA playground. The simpler alternatives are useful for quick experiments. After changing a setting, rerun the NCA section from the target cell onward.

In [ ]:
import base64
import re
from PIL import Image

y, x = torch.meshgrid(torch.linspace(-1, 1, GRID_SIZE), torch.linspace(-1, 1, GRID_SIZE), indexing="ij")

if TARGET_PRESET == "image":
    encoded = re.search(r"const PORTRAIT='([^']+)'", ca_playground).group(1)
    pixels = np.frombuffer(base64.b64decode(encoded), dtype=np.uint8).reshape(-1, 48)
    image = Image.fromarray(pixels).resize((GRID_SIZE, GRID_SIZE), Image.Resampling.LANCZOS)
    target_2d = torch.from_numpy(np.asarray(image, dtype=np.float32) / 255.0)
elif TARGET_PRESET == "circle":
    target_2d = (torch.sqrt(x**2 + y**2) < 0.65).float()
elif TARGET_PRESET == "cross":
    target_2d = ((x.abs() < 0.14) | (y.abs() < 0.14)).float()
else:
    raise ValueError('Choose "image", "circle", or "cross".')

target = target_2d[None, None].to(device)
plt.figure(figsize=(3, 3))
plt.imshow(target[0, 0].cpu(), cmap="gray", vmin=0, vmax=1)
plt.title(f"Training target: {TARGET_PRESET}")
plt.axis("off")
plt.show()

## 2. Build a minimal NCA

Each cell stores one **visible channel** and seven **hidden channels**. One NCA step has the familiar structure:

1. `perceive`: apply fixed local filters to every state channel,
2. `update`: process that local perception with a tiny neural network,
3. add the predicted change to the current state.

The update network uses only `1 × 1` convolutions. All spatial communication therefore happens through the explicitly local perception step.

In [ ]:
class MinimalNCA(nn.Module):
    def __init__(self, state_channels, hidden_size):
        super().__init__()
        self.state_channels = state_channels

        identity = torch.tensor([[0, 0, 0], [0, 1, 0], [0, 0, 0]], dtype=torch.float32)
        sobel_x = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=torch.float32) / 8
        sobel_y = sobel_x.T
        laplacian = torch.tensor([[0, 1, 0], [1, -4, 1], [0, 1, 0]], dtype=torch.float32) / 4
        kernels = torch.stack([identity, sobel_x, sobel_y, laplacian])[:, None]
        self.register_buffer("perception_kernels", kernels.repeat(state_channels, 1, 1, 1))

        self.update_net = nn.Sequential(
            nn.Conv2d(4 * state_channels, hidden_size, kernel_size=1),
            nn.ReLU(),
            nn.Conv2d(hidden_size, state_channels, kernel_size=1, bias=False),
        )
        nn.init.zeros_(self.update_net[-1].weight)

    def perceive(self, state):
        return F.conv2d(state, self.perception_kernels, padding=1, groups=self.state_channels)

    def forward(self, state, return_delta=False):
        perception = self.perceive(state)
        delta = self.update_net(perception)
        new_state = state + delta
        return (new_state, delta) if return_delta else new_state


def make_seed(batch_size=1, row=None, column=None):
    row = GRID_SIZE // 2 if row is None else row
    column = GRID_SIZE // 2 if column is None else column
    state = torch.zeros(batch_size, STATE_CHANNELS, GRID_SIZE, GRID_SIZE, device=device)
    state[:, :, row, column] = 1.0
    return state


model = MinimalNCA(STATE_CHANNELS, HIDDEN_SIZE).to(device)
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"State shape: {tuple(make_seed().shape)}")

## 3. Train the local update rule

For every optimization step, the NCA starts from the seed and runs for the selected number of rollout steps. Only the visible channel is compared with the target. The hidden channels are free to organize useful internal signals.

The final layer starts at zero, so the untrained NCA initially makes no changes.

In [ ]:
BATCH_SIZE = 2
optimizer = torch.optim.Adam(model.parameters(), lr=2e-3)
loss_history = []

model.train()
for iteration in range(TRAINING_ITERATIONS + 1):
    state = make_seed(BATCH_SIZE)
    if RANDOMIZE_ROLLOUT_STEPS:
        rollout_steps = random.randint(*ROLLOUT_STEP_RANGE)
    else:
        rollout_steps = TRAINING_ROLLOUT_STEPS

    for _ in range(rollout_steps):
        state = model(state)

    loss = F.mse_loss(state[:, :1], target.expand(BATCH_SIZE, -1, -1, -1))
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    loss_history.append(loss.item())

    if iteration % 25 == 0:
        print(f"iteration {iteration:3d} | steps {rollout_steps:2d} | loss {loss.item():.5f}")

## 4. Compare with a pretrained model

The live model above demonstrates the training process. The embedded model below has the **same architecture and target**, but was trained for longer with rollout lengths sampled between 24 and 40 steps. It was selected using validation losses at 24, 32, 40, and 64 steps.

In [ ]:
import io
import zlib

PRETRAINED_WEIGHTS_B64 = "eNrtmndUFMu2xoeM5CRBJSkoBiTH6b2RaMAIBkxwAEcBARkYBNRRUAFBlKAoKogBRMUAIqLQVSCIAuaEB0zowSyICRXTAz3nXOWee+667/331q1Zvaq7q76vft1dvWdN7ZnsLibO4UhLc/6pKHFUOL7h/gGBi3km83wFviPDFga7OSlwZv1UYkX9/RcFB/P8BYGLQiNkJoXP44Xz5rkE+gtk+JyhHnwRIy+VbrMwXrg/L6ynj89CXngoLziCL+ovWNRtP9InUhAYHCHjE87ziwwMnucj4IVGLAr3WWwuwxczMvKS6lZHdPf0XcDji3+XyLgFL/IVeH4/KcOX8BLp7mPKl/QS6679wyL5UhP0RAR86SnuHCN3PXcRdzF3MQG/j5F7n+6PmHt3k0xSQA+crIAv58GX91Lt1kWGdV8kzyeUJxhpOjKKF7ggQMBXCOgmCBAP+D6CGV8xQGoCR1zAV/rduce72035+yjf9lW+O6sK+Goe/L5eyr2d/QJ9I/jqP/ma8zUCpNz1BHzNblt3vXi+lrtIPL/fd6P+Av4AD752b0TzPxB1frKy4Ov2IHZz6H1DlP4DUf8HxIHfnQcJ+AYefMNIIX+wV59urU8IT+Db86D5Q761d9/7nmnAHyrkD/v2FBbzwiO6HyB/uLtIhJfMTzz8EUK+ccC3BrleV8wfKeSb/GWTGd9UyDf7yyZzvrmQb/GtKTLCb+RkdynpL2Q28EQ5nJ5t8r+ctBoc9T8n7cj5i8JDfAU+v3O7Oan0mrs9xazHu+zFrVU9t69n+9fe/Tg2//D+fUr6+AYHLggN4YUK3Jws/sL+3xZryx4Ahyj+k+7r4oj+LYAyx+FPAL8YAW9Rz8vm5sT934z7RwkOFAiCeT0M8fCblmT3KJJ/y6DAwZ+igompm5P9/wHgn0eIdfjncxz6e43d7b/vx+LP53v2v7XRXlr8vS/+oPu9jq38+fivyn95/svzX57/rzw9kW/GxDxhbHe869n+LvKJ/xz5zNycOBzdd+bQ8WE/HdXwC/52PRn3n3pBRBrOkIXlgfYqfmbIj1tNWt9uwXNqZTgw41dor6Ukb+WtCj5PEeQ3FGKVeinwOMfogjMaJLdzAV1/UAlFNx4mVaXb0Nd3FY6I57Izi/OY6W5rseMwg1UKNZDQIE1PuWti0JhkcqF8XHlNuzfYNCmTiQ8yMbTtDStMvkIs51aQTQGz6fFHZZBhtY2U70qnQ6qtUT99JkqcEUGRuHhI4HWyk0bshYNbRStPXM2vOF5Xh5rvP4D+eFl0TcmiV4edIeNdIsgWuZFUKTmN4fyWBhfCzdnPBUPpNbkDkKZdR4Y2qtJCu3A4sjKHqiicZJxOpzM8p4cgwe5lW9Xnkc5MeRiEo3CIlT0affyA70cTZk/gYexSOITJ1xWx6aUtzp1HaNvIJfQh3YpZS1dQYY0sFNmJ4a+WRrTPO0WiHtCCs6pX4OwME5zZJAkFSx+wqktmgBhZhzteWNH5a21Aw6KaRienoc6tdGJ3ppnVLlKtdNJIJNNu3mPPNRzEFbPuEN9UV5QxyCOv7qs5fBo8nV6alIQjhTthmlkjdbVMpycu2WJa+20yY/4pSDqeQb982ESvukyHl3SgQ9uVCxXRQzPxXVUV07xFEucdUqosOWCF5y7J4NP2alysIk15M1VIfWg+cdUoRu2rDvhxuQHuPjKBTru/GcMmCjGEmqDiaUn7uvAITM0nYDO4PzyXrCS5rV3EX5BKb9/5jbye9iuc/2KHOVZjcfRLNTR7OZ3uGrgOu9AF2wzD8bf1YrR5rg9eGF0CsgFy1Pr8DbLTQ4HuUGiiw0ss6CQLCsK6J6SwWp6MfnWdDGgfBdOerYfG8hT8LIyAGaXmKP5ZFKWPPGUqOo3ZctlF5JPxcXL6UzS1tL9JhzSYQahcJpzps48c0bwCRoEHiOpUA0hcPoLW6FmhaWsc1KdLwaYoJ9R7tAIjq9Og9fAsetS1HwauuYVqpyXAX3c/HBs3C5Pu94XPw1cyhUEcnOAuIJ0iImS57XAavzeUis02xWPF2fh11WiI3RSOhq+iqeNHGczWY2i8cBZGzE5FhUcJJGt9FQQPbULpMUPQsNiFWX2uwb4jcRuRfKxKJsuvRg3fVVydd850leJN2nnvGjuseiCNN+4HtrF7aHxLA3m1tAiMQyuxNopPTaoKMPN5C3jsrMI903PQiRHAbYn9MPB2OE36YEcn5b1FnTfLqUGZB05pNaYTzRvpji/eJKjkEVafVMKAkePRUFeFaBmuwVndseexUNwhfe42cqqjGk8dbKcLX69B5w1vSZFsDV5Y2klmN8yCQ6m+MHvSWFQPk4P1K40Iu8OHUq81uPPyPvqoOBFGHFQiB1OkHH7Z2U5dj6jTLjjKKsR8JtNnVmBfKTvM7HsauAYUS4uaSO2uSnigy6UfP57A1TEvqeSDe0TxYxnwWteQpbdO0qSZ0rDszTXyW0Q62EqtZqct9EEPyd2k4PYt1GsUp/1vHAfrM6Ow/aIvyhxYSP29BuF1redMV+I0ZI68pIrOmnDlcyz5tESBJjo9wXuhT8li41ckSzUHMy7VQsGd/VhytYs46r/Ft2sO02YrU5qy2hOa2GLQNBwPbSfNKrsWrSKLf+HQh2Gi2KqmBVqSFUzMOF9QnXu9Wx+EYyyG0S752aS9QxIc36dDoY0THbzch/gmCEmlnREtPxSEoYIZWN7ggncdE8mpnZUkf4wuOG+Mw+bLJuj8Iona7/fBaq0k0lm3gVnpoo/2Uo7Yx9cM3QwHkqjmTBLy4QT61Ebjjcc5KFKujg8dduCRdV2w9LM0jnbIgaN2nZBaw6Xm9UvQuSKQlN6Wq2zOH4b9Q0ZgqMVd7invJHAcMR1PLnXG5jQxGl+zDkt3bEet/GOYMluSbB2qSedamaODaR6GZcnRYZEaVGV6Nm2O6MJ3Y+th/6IceumVJ426+558IorU8dUm+i7XGTlv1fDVpS0w9qodfbfjKb2kXQctoq7khNRDtHO1QwmOPTwPPwoJ4zOgWGovas9/yGwUzq+wHjiEOmTsZb3XCsm2i1uI6OE6NsWbJTD6ALFpsaf9hqhgDTkODVYHIfiqDInxSqXnNVLpkwwWZy6TZmYd9qJrSirhxeQL6JjvT7IKl+BBxX20pG87yd6sD+oHUvDxkJXkoYwPnNk6G4VjC2iGIJCa2b4BF2ETRBqrUZuMQJp2Ugu27kwB/ZgHMCJwPzUvuwNU+ITsI7ncnPo+WBWfhVqasjghno/7B2lg+LP1mOyWQ9jsBFxcXEX6OCg52NnuIm+16rF/hoHDDL8duCP2EDmRJUH3bywhbe5NuLD5DhPX3xqDPm9g4mq9qDCyD63efh/3uauBz8o0/Bppj0mSfige6YVngaL/jSHo1iJCzW554+6tK6ngvTnbf1w9NanmYYvMKOQfXcvcmv8rzZW1RJnOy1ibPQ3aQn3o5RXWzOIlJ9E1VAQ3S6RgQ//nJPasMh1C60iKjGKlo6YxXnkchWtu2UJDGtC+v2xGgdYuVC9nyXtzWVrQcBcvrcoH5bu1wITF0PGvMrBc+iaeVa4kNZ/30cDbx7l14+tgVJQEdRlaSILCdTG2eRriyBZ2edfZisiSG/D0XSL6fRiEz+TrsGjUI9L3/hVwDTrWHQ+a6N3RQeg/sxzNa4pZ8bQMOvFWOroem4i+TedIjHAC6r1PJPIfs1AxWQWOrFKBxlIhGeR7FY3eHoM2HysMG+EM0/o0QJCTGL15MBTfHBOlVViJG9Y+Ip+e6KGRVSX1KORiVnI+N3jJuQqt+9X0iN14THPdAscPSeDH3Hmsn8VVdsbYhyTNGTDQUQyiqoIRDMXpQxMdTIJr1GRSCbZEqMGlzilYV/qBol0eJg0cS4pPJIB7hEIl194L4qcLSfsIM3o1s4gapvqj+OMNkN14AMvKs6lq1Uxseh5HrnbYYaptAamrP43bFOQxzzOffmmYT+Nnnid1c9RpR8A8xsU1m4as3Yx99etIc9ghbFxtSd+cXIsXBqbTw3FdVBm1HXafXOZwL+QoHJyE2DWaQ1PcJah4xwoYXHad5hutpfO5ByA3bheJ1nxNePWJVK91KYSsU8FP6ZfwrLwMsv5a0Lq1ghxty6UtdjupROVZOGBvQpR1Usn2HXKk5cgc9B40nOQPTWVXaJSA6bJjcOGiI9qdTaXT9S7SNMEWmDVOHF+O86SnF3JpyPERKJUaTX/9okdhySRaFJfAvTmjkamKEdCAEmv6ZvV5aHCeA/qFPhjxUpVqPc3FCRNcSI1VOBWZP49o8qRBTs0EzVY9B+utldS0ywa2vY0iOQvH0LqxRWTbsT0omcyhFfrilatSrHBAyFes7KtKghSNaH1lCaqtHo8zbKTx5ENtFFGKRkVGCcWfWpOgOcmk4XwuVnnnYF1YHTs9VLxSwdGTZBeNxqomafxizEWjiVuQf/ArrJjSCaQgjritmIEryqciV3IofAiVp1MK90Fc9HKur2w8SdKeSYfwytF6fDzKORwkd2JSaF2kBm7I24xThrgTj+fLacJbFXzZMgC5y8Nw4ZSpdEVjGvFtqMEhwq+kLuYRhm05j1Ii1WAQsxtGnfeh2XGT0dLhZDdPHypjv5DKFnXBI5NG3CffCq1LVuLt8o34LNKTsOPCaFH0NPywqxpL9bPw09PZZEd0EfX0v0Ok8y4SdtJ8ciI4i277VYO+2T8ajQNV8dDVOPro2TiS+9CD3B88jCaPiIIAxQ5Qt2qu+FA+ES8RZxrdsoye315LPkc5osqpz9SmLJGWXSliopcEMZmrntFPZeupc+og+u6UP6yYkAf8kAQ6xb4dqtb2o4EBm1DW+hpj2bwOJx7RYW9678W0MC/2S4cFHspWJsoB5zBWoalCVawQibcTCHdXU5HB92ln8ku8dGosVdbuhDHzdeHlrO1MiMEloMQVOzJn4MmLW/DQqWGo+FIZrGMDaKHfNdzh/5LdX9rKyt99Ssdd5EKMuTsVLXzIfOSkk76KR0nZNUs66fk+zPXSwjTZUZidspGRM7LDZ2a6BOUJSVhwo/xySyQ+X9pB/awP48QxJczm9H64VDcDS2VK4Fh0C0mSS0GXfXz0qwmFqj590Et5FbinbqJXYoX4tlEP19cl03WfJZnhEs/p2WIf6uvSj+oZrWAca/zou4SC8qoz3pChHUiuz0kDo6XaVMJPDT/ves0Mi+og9tFDqLWsNazP80f7ZlEyTe4yqGnLY1GsO5Eb3E4mnDBl5g+UpLln7Cl5O5b6lLYSz2UMZru0MwuGHGGeeI7GU+PbGB31Onyu+Y4uS34ImZ6J1HqoDNQWzITJBrns2sS79HVrME3KLmaDncVg6upm+t5aFs+Lj8NtqllQWiTqsHXOe0g9p1w58v4zMny6Nd3Qbz0Vzv8EynlPmRE3jeloq2fkZUQjVM+dwu5+fJzKmWzG9hUF9HT3b5hrr+9gmPo5qM5j6KiJgHG1u3HqPEW0WT0cJzZ3kJu7NyF7NAdEovfj1h3rwWxbJe5ZVwdSsx3QaR4XNs/2J8KjcbRG2oyerxhLcxN88eDps2g5hmJEkiG9bhKOea4VuF5L1qHSfBzJq3LHiCBnZsH46zBF3AOpfgVaywRjW9kXunCXJu26fxYkhJuxz8Bc6vd+A3Aojxae1iDXzqYS319GoE/6BCya3R8la9RJo04XszZez2GM1TgMqBpQee0lH+zrrlLjAaLUangyvps0ANv6f7avm/MKfEqG05V8FxwQthnPSCei6YP1tOS6NkptUMLC8WtQ5JY/sazXIE6vg1F92H4seL2WHJXRx1/6TsDH8zaiZLAQUlZvYm7tXQvCC+dh0d43RGZzJtjVHsXDo6aSCbMs6Y7fHOFpMgdvNxTjCQ0H1A6VpHGWE5npMsewUMuVyRJtgLF39+DbsmHEVC6DFIY0kL2VTagYMRLiHb5S0cXl7HRuLqObyAeFoAZSarsGisdSnBpox4p+HUoF8zeV3xU91x0bY2hO7RYaoDeHDEziOlxuvQLjvftWxjxNpN4mX2mHxBny9qsHinZNo9MMNkBkfTLa3i+gNR4y9G5CEabP+cjVfb2Z9qwfbK6dL8FR+pYy+U/WD8x71g+m1gZzvfsdJveu+cHXs+J40rm0O/6tYu0t5nAPbPGEDuc3gKuk4YX7QKqipEIcTd+TrXIXyZfuWLMoP4NsTJNgy2IPQdHyw2DarAnbwy+zIi4LycjsU8ynwzLUb3khWaraCG5lImyF6E7yK8+XuEo8J8ulM4lLWl/o4detefA4tmdFhPMf8Vv08L9bowRU1B781aWAWJvDrdeUvdXiyp18b1YFe4sLHWZG3Mv3TzAVw5+RqY5ZREXsNGicbGTnXlvI6B9cSXKGLoNB13KZnTfOMvmibWAgm0GKswcBmlYyJ/0OMC6C6yDXUs+1OdTASliUk6sTusDcRp3UTzFgH5gXsUlbAtCicypd7iFFCq9tYO/FFELUIpHy4sXx3CdTJrE6bqOhZWkuN6CxnYScsMCcinxIgkGoLm4JOpoGmFoagMYnNjCv3eXIhaXO9HKmKXNJw5AJkeqLPM+P7AO38zDk4Ajm+IOLkPBLM2ktTMP3G0vZ2O7fEh22mWA1OwnUpd4C83Ed+zRLj/3Yrok+YRSunr1AODdfsKubH0Hj1DgQmB0hBu8YZnKUNq54c5fcNGSZteqdMG/URKy9NxlGNt1nJRNmM/pj+xOeKRc6W7ej0uMFVMS4gOi03+BG3upLjhvPAXZrJuy2sMWNU0XI5+oENBOWQtmUc2QqdjEuhhx40fUFLCQiYdvZQKioV2OKZ3yFZ6p90NIniYYpVJF9y+OYWAtDutyujXmSvBd0k7NQJJwLRadyiNhgwrYMd0ADqdOQN1gVhmt7UZ3QQnJ+eCb47FFl+nnzGYGzPs6MH03qX+0ju5/q455POXSqQBSUXB7B+gJP5twiXeDUm5D1fHW86DoWGQcvUvBLOnvfI5q0T2uHJ9MrYZPDKOxnxsKupjNkuUM4HWuoD4PqEvES20DED3pC0bBR9J7MYrpd2QbNzjhioYo8eKvchYneg2jHBdlK8ZiPxHq7Nyx5M4fGyluhmegadkjybjJuiTzdE3EENLu8SYe2BzkzX4eeCH3A3hiuS/SelZE7XRvhyJfd7DiLudSqK4YYLLPC+6vKUS3an7mfWkUGhVhBdMAntvKEHry6HIX23KH04FJNLD/nQHQ+qYO9ZDXs/nIIry38FdIPGWJdQyUeXcBFa29XunHqDm5xswERkZLCnXfLmRnu6ozWNmmyKGgHc0h3Bju8XpTKP3oB5wuNSP9Dg+gp7UlM7t69mHF3Bt5QOMRET5AmEttNiWxDIXHcIAIyJtkYLZlO+wTnMO+NNtt7u+2DkrB88PReS/xyO+DJOzUs3/AGC5cMo8/SXzApEgXQYZXPFU5NhDZPAjYJhnBisTi6ZW+F/lPiwW+mLfeG2itmwVJjsniFHFGs8oTEMa/gkmkKuyszDDQu9ofO2S50eaAlDu5cS8YfqSPa09RJVHI8ZNbxuG+CFkKLahtTO6CejN9XwrQdASJW++xb/OAm5E7uDg4czt+unypynP+MH3+m5Bz+L7mrnmIh00NwefuCaf8+f6bDMf9HAu9bCIvghQf6Bgcu8f2WmQ6c5+Y08j8EMrO2s7axtja3sbGwsLS0sbW1tbIytTO3MLMzN7WxMje1szG3s7CztuuhDPaqFjPq5jD6RinSA/sn54/pTaV/Iu+dh++t/jGBqfGTslSU868Sor1NfkxC9vs58SjG+ZvMZ2+fHxOJyj/5ZPzg82cCs7f8x9V4hZ/k6uKcXrnH3tofv4l/1ir16aU166398VvwZ+1krV5a897aH9+AXuP266W16K39ce4q/jyuDqf3G9Nb/OOU0vlJfOUH8V9P9snukpIj/ng1xIz/VMr0qleJfq+jdX/Hcpfs+TsA56P+9+PvWXMJye8amW/9v/f9H3dqSCc="
PRETRAINED_ITERATION = 1400

checkpoint_bytes = zlib.decompress(base64.b64decode(PRETRAINED_WEIGHTS_B64))
pretrained_state_dict = torch.load(
    io.BytesIO(checkpoint_bytes), map_location=device, weights_only=True
)
pretrained_model = MinimalNCA(STATE_CHANNELS, HIDDEN_SIZE).to(device)
pretrained_model.load_state_dict(pretrained_state_dict)
pretrained_model.eval()
print(f"Loaded pretrained model from iteration {PRETRAINED_ITERATION}.")

In [ ]:
model.eval()
with torch.no_grad():
    trained_state = make_seed()
    pretrained_state = make_seed()
    for _ in range(TRAINING_ROLLOUT_STEPS):
        trained_state = model(trained_state)
        pretrained_state = pretrained_model(pretrained_state)

fig, axes = plt.subplots(1, 4, figsize=(13, 3))
axes[0].plot(loss_history)
axes[0].set(title="Training loss", xlabel="Iteration", ylabel="MSE")
axes[1].imshow(target[0, 0].cpu(), cmap="gray", vmin=0, vmax=1)
axes[1].set_title("Target")
axes[2].imshow(trained_state[0, 0].clamp(0, 1).cpu(), cmap="gray", vmin=0, vmax=1)
axes[2].set_title(f"Just trained ({TRAINING_ROLLOUT_STEPS} steps)")
axes[3].imshow(pretrained_state[0, 0].clamp(0, 1).cpu(), cmap="gray", vmin=0, vmax=1)
axes[3].set_title(f"Pretrained ({TRAINING_ROLLOUT_STEPS} steps)")
for ax in axes[1:]: ax.axis("off")
plt.tight_layout()
plt.show()

# Explore the trained NCA interactively

The app below runs the model you just trained. Follow it one update at a time, add seeds by clicking, try different initial states, and switch between the visible state, every hidden channel, and the update magnitude.

In [ ]:
import json

# Export both models to the browser app.
def export_model(selected_model):
    selected_model.eval()
    return {
        "kernels": selected_model.perception_kernels[:, 0].flatten(1).detach().cpu().tolist(),
        "weight_1": selected_model.update_net[0].weight[:, :, 0, 0].detach().cpu().tolist(),
        "bias_1": selected_model.update_net[0].bias.detach().cpu().tolist(),
        "weight_2": selected_model.update_net[2].weight[:, :, 0, 0].detach().cpu().tolist(),
    }

payload = {
    "size": GRID_SIZE,
    "channels": STATE_CHANNELS,
    "hidden_size": HIDDEN_SIZE,
    "target": target[0, 0].detach().cpu().flatten().tolist(),
    "models": {
        "live": export_model(model),
        "pretrained": export_model(pretrained_model),
    },
}

nca_app = r"""
<!doctype html><html><head><meta charset="utf-8"><style>
*{box-sizing:border-box}body{margin:0;font-family:system-ui,sans-serif;color:#202124}.app{max-width:960px;margin:auto;padding:14px;border:1px solid #dadce0;border-radius:14px;background:#f8f9fa}.top,.row,.stats{display:flex;gap:8px;align-items:center;flex-wrap:wrap}.top{margin-bottom:12px}.layout{display:grid;grid-template-columns:minmax(340px,1fr) 250px;gap:14px}.canvas-wrap{background:#222;border-radius:10px;padding:8px;display:flex}.canvas-wrap canvas{width:100%;aspect-ratio:1;cursor:crosshair;touch-action:none}.card{background:#fff;border:1px solid #dadce0;border-radius:10px;padding:12px;margin-bottom:10px}.card h3{font-size:15px;margin:0 0 9px}button,select,input{font:inherit;border:1px solid #bdc1c6;border-radius:7px;background:#fff;padding:6px 8px}button{cursor:pointer;font-weight:600}.primary{background:#333;color:#fff!important}.wide{width:100%}label{font-size:13px;font-weight:600}.small{font-size:12px;color:#5f6368;line-height:1.4}.stats{justify-content:space-between;margin-top:8px;font-size:13px}@media(max-width:720px){.layout{grid-template-columns:1fr}}
</style></head><body><div class="app"><div class="top"><button id="start" class="primary">▶ Start</button><button id="step">Step</button><button id="reset">Reset</button><button id="clear">Clear</button><label>Speed <input id="speed" type="range" min="1" max="12" value="5"></label></div><div class="layout"><div><div class="canvas-wrap"><canvas id="canvas" width="640" height="640"></canvas></div><div class="stats"><span id="generation">Generation 0</span><span>Click to add a seed · right-click to erase</span></div></div><div><div class="card"><h3>Model</h3><select id="modelChoice" class="wide"><option value="live">Just trained</option><option value="pretrained">Pretrained</option></select><div class="small" style="margin-top:8px">Same architecture and target, different training duration.</div></div><div class="card"><h3>Initial state</h3><select id="initialPreset" class="wide"><option value="center">Single seed</option><option value="two">Two seeds</option><option value="four">Four seeds</option><option value="random">Random seeds</option><option value="empty">Empty grid</option></select><button id="load" class="wide" style="margin-top:8px">Load</button></div><div class="card"><h3>Visualization</h3><select id="channel" class="wide"></select><div id="description" class="small" style="margin-top:8px"></div></div><div class="card small"><b>Try this</b><br>Advance one step at a time. Then compare the same seed using both models.</div></div></div></div>
<script>
const M=__MODEL__,N=M.size,C=M.channels,H=M.hidden_size,P=N*N,canvas=document.getElementById('canvas'),ctx=canvas.getContext('2d');let W=M.models.live,state=new Float32Array(C*P),initial,lastDelta=new Float32Array(C*P),generation=0,running=false,timer=null,drawing=false;const at=(c,r,x)=>c*P+r*N+x;
function seed(r,c,on=true){for(let ch=0;ch<C;ch++)state[at(ch,r,c)]=on?1:0}function copy(a){return new Float32Array(a)}
function loadInitial(){pause();state.fill(0);let p=initialPreset.value,m=Math.floor(N/2);if(p==='center')seed(m,m);if(p==='two'){seed(m,m-6);seed(m,m+6)}if(p==='four'){seed(m-6,m-6);seed(m-6,m+6);seed(m+6,m-6);seed(m+6,m+6)}if(p==='random')for(let i=0;i<8;i++)seed(3+Math.floor(Math.random()*(N-6)),3+Math.floor(Math.random()*(N-6)));initial=copy(state);lastDelta.fill(0);generation=0;draw()}
function update(){let perception=new Float32Array(C*4*P);for(let ch=0;ch<C;ch++)for(let k=0;k<4;k++){let pc=ch*4+k,ker=W.kernels[pc];for(let r=0;r<N;r++)for(let c=0;c<N;c++){let sum=0;for(let dr=-1;dr<=1;dr++)for(let dc=-1;dc<=1;dc++){let rr=r+dr,cc=c+dc;if(rr>=0&&rr<N&&cc>=0&&cc<N)sum+=ker[(dr+1)*3+dc+1]*state[at(ch,rr,cc)]}perception[pc*P+r*N+c]=sum}}let next=new Float32Array(state.length),hidden=new Float32Array(H);for(let pix=0;pix<P;pix++){for(let h=0;h<H;h++){let z=W.bias_1[h];for(let i=0;i<C*4;i++)z+=W.weight_1[h][i]*perception[i*P+pix];hidden[h]=Math.max(0,z)}for(let ch=0;ch<C;ch++){let d=0;for(let h=0;h<H;h++)d+=W.weight_2[ch][h]*hidden[h];lastDelta[ch*P+pix]=d;next[ch*P+pix]=state[ch*P+pix]+d}}state=next;generation++;draw()}
const clamp=x=>Math.max(0,Math.min(1,x));function color(v,mode,scale){if(mode==='visible'||mode==='target'){let q=Math.round(255*clamp(v));return[q,q,q]}if(mode==='delta'){let q=clamp(v/scale);return[Math.round(255*q),Math.round(180*q*q),Math.round(30*q*q)]}let q=clamp(Math.abs(v)/scale);return v>=0?[255,Math.round(255*(1-q)),Math.round(255*(1-q))]:[Math.round(255*(1-q)),Math.round(255*(1-q)),255]}
function draw(){let selected=channel.value,mode=selected==='target'?'target':selected==='delta'?'delta':selected==='0'?'visible':'hidden',values;if(mode==='target')values=M.target;else if(mode==='delta'){values=Array.from({length:P},(_,i)=>{let s=0;for(let ch=0;ch<C;ch++)s+=lastDelta[ch*P+i]**2;return Math.sqrt(s)})}else values=state.slice(Number(selected)*P,(Number(selected)+1)*P);let scale=Math.max(...Array.from(values,v=>Math.abs(v)),1e-6),image=ctx.createImageData(N,N);for(let i=0;i<P;i++){let rgb=color(values[i],mode,scale),j=i*4;image.data[j]=rgb[0];image.data[j+1]=rgb[1];image.data[j+2]=rgb[2];image.data[j+3]=255}let off=document.createElement('canvas');off.width=off.height=N;off.getContext('2d').putImageData(image,0,0);ctx.imageSmoothingEnabled=false;ctx.drawImage(off,0,0,canvas.width,canvas.height);generationEl.textContent=`Generation ${generation}`;description.textContent=mode==='visible'?'Visible channel (fixed grayscale range 0–1).':mode==='target'?'Training target.':mode==='delta'?'Magnitude of the most recent update across all channels.':'Hidden channel (blue = negative, red = positive; range normalized per frame).'}
function pause(){running=false;clearInterval(timer);start.textContent='▶ Start'}function toggle(){running=!running;start.textContent=running?'Ⅱ Pause':'▶ Start';clearInterval(timer);if(running)timer=setInterval(update,1000/Number(speed.value))}function position(e){let b=canvas.getBoundingClientRect();return[Math.floor((e.clientY-b.top)*N/b.height),Math.floor((e.clientX-b.left)*N/b.width)]}function paint(e){let[r,c]=position(e);if(r>=0&&r<N&&c>=0&&c<N){seed(r,c,e.button!==2);initial=copy(state);generation=0;lastDelta.fill(0);draw()}}
const generationEl=document.getElementById('generation');channel.innerHTML='<option value="target">Training target</option><option value="0">Visible channel</option>'+Array.from({length:C-1},(_,i)=>`<option value="${i+1}">Hidden channel ${i+1}</option>`).join('')+'<option value="delta">Update magnitude</option>';channel.value='0';canvas.oncontextmenu=e=>e.preventDefault();canvas.onpointerdown=e=>{e.preventDefault();pause();drawing=true;canvas.setPointerCapture(e.pointerId);paint(e)};canvas.onpointermove=e=>{if(drawing)paint(e)};canvas.onpointerup=()=>drawing=false;start.onclick=toggle;step.onclick=()=>{pause();update()};reset.onclick=()=>{pause();state=copy(initial);lastDelta.fill(0);generation=0;draw()};clear.onclick=()=>{pause();state.fill(0);initial=copy(state);lastDelta.fill(0);generation=0;draw()};document.getElementById('load').onclick=loadInitial;modelChoice.onchange=()=>{W=M.models[modelChoice.value];loadInitial()};channel.onchange=draw;speed.oninput=()=>{if(running){pause();toggle()}};loadInitial();
</script></body></html>
"""
nca_app = nca_app.replace("__MODEL__", json.dumps(payload))
display(HTML(f'<iframe srcdoc="{escape(nca_app, quote=True)}" width="100%" height="735" style="border:0"></iframe>'))

## Take-away

- Repeated **local interactions and updates** can create coherent global behavior without centralized control.
- With deep learning, we do not have to design the local rule by hand. We can specify the desired global outcome through training data and optimize a shared local update rule with gradient descent.
- Visualizing hidden channels and updates over time offer insights into how the NCA organizes and transforms information.